[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Jibby2k1/SPS_Curriculum/blob/main/Intro_FPGA/Intro_FPGA.ipynb)


**Content Produced by UF Signal Processing Society**

**Authors: Raul Valle & Contributors**

# Intro to FPGA

> ⚠️ **Draft — code not machine-verified.** Verilog below is shown in fenced blocks and has not been simulated by an automated check. Before teaching, an instructor should run each module + testbench once in a simulator (Icarus Verilog is free: `iverilog -o sim tb.v mod.v && vvp sim`, or use [EDA Playground](https://edaplayground.com/) in a browser). Remove this banner after that pass.

Where [GPU workshops](../Intro_GPU/README.md) parallelize *software* across fixed hardware, an FPGA lets you **build the datapath itself**: your filter becomes wires, registers, and multipliers with sample-rate throughput and microsecond-class latency. This is how radar front-ends, SDRs, and instrument DSP actually ship.

## 0. Introduction

An FPGA (*Field-Programmable Gate Array*) is a sea of small configurable pieces:

- **LUTs** (look-up tables) — implement any small boolean function;
- **Flip-flops** — 1-bit registers; state lives here between clock edges;
- **DSP slices** — hard multiply-accumulate blocks (your FIR taps);
- **Block RAM** — on-chip memory; **routing fabric** — programmable wiring connecting it all.

"Programming" an FPGA = describing hardware in an HDL, then letting tools map it onto these resources.

## 1. Pre-requisites

- [Intro to C](../Intro_Programming/Intro_C.ipynb) — bits, twos-complement, thinking in memory.
- [Filter Design](../Intro_DSP/Filter_Design.ipynb) — Session 3 implements that FIR here.
- Tools: any simulator (Icarus/EDA Playground) for S1–S3; a dev board (e.g. Basys 3, iCEBreaker) + vendor toolchain only for S4.

---
### 🕐 Session 1 of 4 — *What Is an FPGA?* (~35 min)
**Goal:** LUTs, flip-flops, DSP slices; the FPGA vs CPU vs GPU trade-space.
**Feeds into:** Session 2 (HDL basics).

---

<details><summary>🎓 <b>Teacher notes — Session 1: What Is an FPGA?</b></summary>

**This notebook is a draft — the banner is load-bearing, and this workshop needs it more than most.** Every Verilog block here is unsimulated; an instructor must run each module + testbench in Icarus Verilog or EDA Playground before teaching, since hardware description code has a much higher "looks plausible but doesn't do what you think" rate than software.

**The core reframe for the whole workshop is "time vs. space" — get this to land before touching any syntax.** A CPU has one (or a few) execution units it reuses instruction-by-instruction; an FPGA dedicates *separate physical hardware* to every operation, all active simultaneously, every clock cycle. The 64-tap-filter example in the intuition cell is the concrete version: a CPU loops 64 times through one multiplier, an FPGA has 64 multipliers running at once. This is *why* a 200 MHz FPGA can beat a 5 GHz CPU on streaming DSP — clock speed isn't the whole story once parallelism this wide is on the table.

**The CPU/GPU/FPGA trade-space table is worth connecting explicitly to workshops students have already seen or will see next:** GPU parallelizes *software* across many identical cores (Intro to GPU Systems); FPGA builds custom hardware *per algorithm*. The deterministic-latency row is the practical reason radar and instrument DSP reach for FPGAs specifically — a GPU's latency depends on batching and scheduling (jittery, per the OS workshop's scheduler discussion), while an FPGA's latency is a fixed number of clock cycles, known at design time, not measured at runtime.

**Set expectations on development cost honestly, since it's the real reason FPGAs aren't used everywhere:** "dev effort: high" in the table isn't a throwaway row — verifying a hardware design is fundamentally harder than testing software, because a bug found after synthesis costs a full re-run of place-and-route (often minutes to hours), not a quick re-run of a script.
</details>

## 2. The Trade-Space

💡 **Intuition.** A CPU executes your algorithm *over time* — one flexible pipeline reused every instruction. An FPGA lays your algorithm out *in space* — every operation gets its own silicon, all active every clock cycle. That's why a modest 200 MHz FPGA can out-throughput a 5 GHz CPU on streaming DSP: a 64-tap filter does 64 multiplies *simultaneously*, forever, with no instruction fetch at all.

| | CPU | GPU | FPGA |
|---|---|---|---|
| Flexibility | highest | high | rebuild to change |
| Latency | µs–ms, jittery ([OS scheduling!](../Intro_Host_Prog/Intro_OS/Intro_OS.ipynb)) | high (batching) | **deterministic, cycles** |
| Throughput/W on streaming DSP | low | mid | **highest** |
| Dev effort | low | mid | high |

---
### 🕐 Session 2 of 4 — *HDL Basics* (~40 min)
**Goal:** modules, combinational vs sequential logic; simulate a counter with a testbench.
**Builds on:** Session 1. &nbsp; **Feeds into:** Session 3 (a hardware FIR).

---

<details><summary>🎓 <b>Teacher notes — Session 2: HDL Basics</b></summary>

**"Unlearning sequential-execution instinct is the entire difficulty of week one" — believe the intuition cell and plan the session around it.** Every student arrives thinking of code as a sequence of steps; Verilog isn't that. `assign y = sel ? b : a;` isn't "if sel then y=b" evaluated once — it's a physical wire that continuously, instantaneously reflects whatever `a`, `b`, `sel` currently are. Say explicitly: nothing in this file "runs" in the software sense; every statement describes a piece of hardware that exists all the time.

**Combinational vs. sequential is the one distinction to over-teach.** Combinational logic (the `mux2` example) has no memory — outputs are a pure function of current inputs, changing instantly (well, after gate delay) whenever inputs change. Sequential logic (the `counter` example) only changes at a clock edge (`always @(posedge clk)`) — between edges, the flip-flops hold their value no matter what the inputs do. The `<=` (non-blocking assignment) inside a clocked block is worth flagging as *the* Verilog gotcha: it means "schedule this update to happen at the end of the current time step," not "assign immediately" — using blocking `=` in sequential logic is a classic bug that simulates correctly but synthesizes into different hardware than intended.

**The testbench is not the design — say this before students confuse the two.** `tb_counter` is pure simulation scaffolding (`$dumpfile`, `$display`, `$finish` are simulator directives, never synthesized into hardware); its job is to drive the clock, apply reset, and check behavior. "Hardware debugging is waveform reading" is worth taking literally: have students actually open the `.vcd` in GTKWave (or EDA Playground's built-in viewer) and watch `count` increment on each rising edge after reset releases — that visual is what makes "everything happens at once, driven by the clock" concrete instead of abstract.

**If a student asks why `count` isn't exactly 20 after 200 ns at 100 MHz:** walk the arithmetic — a 100 MHz clock has a 10 ns period, reset releases at `#12` (partway through a cycle), so the exact count at `#200` depends on which edge reset released on relative to the display; "~20" in the comment is deliberately approximate for exactly this reason.
</details>

## 3. Verilog: Describing, Not Instructing

💡 **Intuition.** HDL code is not a program — it's a **circuit diagram in text**. Every `assign` is a wire that exists *always*; every clocked `always` block is a row of flip-flops. Nothing "runs top to bottom"; everything happens at once. Unlearning sequential-execution instinct is the entire difficulty of week one.

**Combinational** logic (no memory — outputs follow inputs like wiring):

```verilog
module mux2 (input  wire a, b, sel,
             output wire y);
  assign y = sel ? b : a;      // a physical multiplexer, not an "if that runs"
endmodule
```

**Sequential** logic (state changes only on the clock edge):

```verilog
module counter #(parameter W = 8) (
    input  wire         clk, rst,
    output reg  [W-1:0] count
);
  always @(posedge clk) begin
    if (rst) count <= 0;
    else     count <= count + 1;   // <= is the *non-blocking* clocked assignment
  end
endmodule
```

**Testbench** — the simulation driver (never synthesized):

```verilog
`timescale 1ns/1ps
module tb_counter;
  reg clk = 0, rst = 1;
  wire [7:0] count;
  counter dut (.clk(clk), .rst(rst), .count(count));

  always #5 clk = ~clk;                 // 100 MHz clock
  initial begin
    $dumpfile("counter.vcd"); $dumpvars(0, tb_counter);
    #12 rst = 0;                        // release reset off the edge
    #200 $display("count = %d (expect ~20)", count);
    $finish;
  end
endmodule
```

Run: `iverilog -o sim tb_counter.v counter.v && vvp sim` — then open `counter.vcd` in GTKWave and
*look at the waveform*. Hardware debugging is waveform reading.

---
### 🕐 Session 3 of 4 — *A Hardware FIR Filter* (~40 min)
**Goal:** implement the [Filter Design](../Intro_DSP/Filter_Design.ipynb) FIR as fixed-point hardware; understand pipelining.
**Builds on:** Session 2.

---

<details><summary>🎓 <b>Teacher notes — Session 3: A Hardware FIR Filter</b></summary>

**This session is where Sessions 1–2's abstractions pay off in a design students already understand from the DSP side — lean on that familiarity hard.** They've built this exact FIR in [Filter Design](../Intro_DSP/Filter_Design.ipynb); the job here is purely "same math, different substrate." The tapped-delay-line intuition maps directly: each delay stage is a flip-flop, each tap's multiply is a DSP slice, the running sum is an adder tree — nothing conceptually new, just a hardware noun for each software noun.

**Fixed-point is the genuinely new material, and it deserves real time — this is where floating-point intuition actively misleads.** There's no float unit in typical FPGA fabric, so taps get scaled by $2^{15}$ and truncated to 16-bit integers (Q1.15: 1 sign bit, 15 fractional bits). Walk the scaling explicitly: `H0 = 3277` represents $3277 / 32768 \approx 0.1000$ — close to, but not exactly, the ideal 0.1 the Python design produced. That rounding error is real and compounds: **"quantizing taps moves the stopband floor" is not a caveat to skim past** — it's the single most important practical fact in FPGA DSP, and it's exactly why the intuition cell insists on checking the quantized response in Python (re-run `freqz` on the *rounded* taps) before committing to silicon.

**The `>>> 15` arithmetic right-shift at the end is worth deriving, not just accepting.** A Q1.15 sample times a Q1.15 tap produces a Q2.30 product (fractional bits add); shifting right by 15 converts back down to Q1.15 for output. Getting this shift amount wrong is a classic fixed-point bug — too few bits and you clip/overflow, too many and you lose precision or get systematic gain errors.

**The verification strategy in the closing paragraph is the professional habit worth internalizing as a rule, not a suggestion:** generate a *golden* reference output in Python using the *same quantized taps* the hardware uses, then assert the testbench's output matches sample-for-sample. "Looks about right on a waveform" is not verification — bit-exact agreement with a known-good software model is. This is the hardware analogue of the debrief discipline used throughout this curriculum: don't trust a result until you've checked it against ground truth.
</details>

## 4. The FIR, in Silicon

💡 **Intuition.** An FIR filter is *born* hardware-shaped: the tapped delay line is a shift register (flip-flops), each tap a DSP-slice multiply, the sum an adder tree. And since there's no float unit in the fabric, taps become **fixed-point** integers: scale by $2^{15}$, multiply, shift back — Q1.15 arithmetic. Quantizing taps moves the stopband floor; check the quantized response in Python *before* burning it into silicon.

```verilog
// 4-tap FIR, Q1.15 coefficients, one output per clock (transposed form)
module fir4 (
    input  wire               clk, rst,
    input  wire signed [15:0] x_in,     // Q1.15 sample
    output reg  signed [15:0] y_out
);
  // taps from your Python design, scaled: round(h * 2^15)
  localparam signed [15:0] H0 = 16'sd3277,  H1 = 16'sd13107,
                           H2 = 16'sd13107, H3 = 16'sd3277;   // ≈ [0.1 0.4 0.4 0.1]

  reg signed [33:0] acc [0:3];          // 16x16 products need 32 bits + growth
  integer i;
  always @(posedge clk) begin
    if (rst) begin
      for (i = 0; i < 4; i = i + 1) acc[i] <= 0;
      y_out <= 0;
    end else begin
      acc[3] <= x_in * H3;                       // transposed FIR: partial sums
      acc[2] <= x_in * H2 + acc[3];              //   flow through registers —
      acc[1] <= x_in * H1 + acc[2];              //   pipelining is built in
      acc[0] <= x_in * H0 + acc[1];
      y_out  <= acc[0] >>> 15;                   // Q2.30 → Q1.15 (arithmetic shift)
    end
  end
endmodule
```

Verification strategy (the professional habit): generate a test input **and golden output in
Python** with the *same* quantized taps, feed the input to the testbench, and assert the hardware
matches sample-for-sample. Your notebook is the reference model; the waveform is the proof.

---
### 🕐 Session 4 of 4 — *Toolchain & Deployment* (~35 min)
**Goal:** synthesis → place & route → timing closure; blink a real board.
**Builds on:** Sessions 2–3.

---

<details><summary>🎓 <b>Teacher notes — Session 4: Toolchain & Deployment</b></summary>

**This session is optional-scope if a dev board isn't available — say so up front.** Sessions 1–3 need only a simulator; Session 4 needs a physical board (Basys 3, iCEBreaker, etc.) and a vendor toolchain. If hardware isn't on hand, walk the four-stage flow conceptually and treat "blink an LED" as a take-home exercise rather than a live demo.

**The timing-closure analogy to the uncertainty-principle trade-offs from Foundations of Signal Processing is worth making explicit, not just alluding to** — it's a genuinely useful mental hook: just as time/frequency resolution trade against each other, combinational depth (how much work happens per clock cycle, i.e. how many gates a signal passes through before the next register) trades against maximum clock frequency. More logic between registers means longer propagation delay means a slower maximum clock. This is *why* the transposed FIR from Session 3 was designed the way it was — pushing registers between every addition (rather than one long combinational adder chain) is exactly the "pipeline more to buy frequency" move described here, applied preemptively.

**Negative slack is worth defining precisely, since "timing failed" alone doesn't teach anything:** slack is (clock period) − (longest register-to-register path delay); negative slack means some path is too slow for the chosen clock, and the design will glitch unpredictably if used as-is — this is not a warning to ignore, it's a hard correctness bug that just happens to not show up in simulation (which doesn't model real gate/wire delay by default).

**Close on the full-circle framing from the conclusion:** this workshop traced one FIR from `scipy.signal.firwin` (Filter Design) through quantization (Session 3) to a synthesizable, timing-closed netlist (this session) — the same filter, three representations, one continuous design flow. That end-to-end thread is worth naming explicitly as the takeaway, not just the individual stages.
</details>

## 5. From Text to Silicon

The flow every vendor tool implements:

1. **Synthesis** — Verilog → netlist of LUTs/FFs/DSPs.
2. **Place & route** — assign each element a physical site; find wiring.
3. **Timing analysis** — is every register-to-register path faster than the clock period? If not (*negative slack*), pipeline more or slow the clock. This is where the transposed FIR's built-in registers pay off.
4. **Bitstream** — the configuration file loaded onto the chip.

💡 **Intuition.** *Timing closure* is the hardware version of the [uncertainty principle trade-offs](../Intro_DSP/Foundations_of_Signal_Processing_1.ipynb) you've seen all curriculum: combinational depth (work per cycle) trades against clock frequency (cycles per second). Pipelining buys frequency with latency.

First-board checklist: vendor tool (Vivado for Basys 3; open-source yosys+nextpnr for iCE40), the board's constraint file mapping pins, and the traditional first design — the counter from Session 2 driving LEDs.

## 6. Conclusion

FPGAs lay algorithms out in space: LUTs and flip-flops describe logic, HDL describes circuits (not steps), fixed-point FIRs map perfectly onto DSP slices, and timing closure is the price of speed. You now know the full path from `scipy.signal.firwin` to a bitstream.

---
## Where next

- [Filter Design](../Intro_DSP/Filter_Design.ipynb) — design and quantize the taps you'll synthesize.
- [Intro to GPU Systems](../Intro_GPU/README.md) — the other acceleration path, compared honestly in Session 1.
- [Intro to C](../Intro_Programming/Intro_C.ipynb) — the software still driving the FPGA from the host side.